# Mew3D — Remote Texture Server (Colab free T4)

Runs **only the texture stage** on Colab's 16GB T4. Your PC still does mesh generation (it's fast there); only the slow painting step comes here, where there's enough VRAM to run it without CPU offload.

**How to use**
1. Runtime → Change runtime type → **T4 GPU**
2. Run the 3 cells in order (first run ~8 min: installs + ~6GB models)
3. Cell 3 prints a **public URL**. Paste it into your PC's `.env` as `MEW3D_TEXTURE_URL`, or give it to Claude Code.
4. Keep this tab open while generating. Colab disconnects after ~90 min idle.

If the URL is unreachable, Mew3D automatically textures locally instead — nothing breaks.

In [ ]:
# 1) Install
!pip -q install hy3dgen==2.0.2 fastapi uvicorn python-multipart trimesh pymeshlab xatlas
!git clone -q --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/h3d

# the texture stage needs a CUDA rasterizer. Colab has nvcc, so build it for this GPU.
import subprocess, torch
cc = torch.cuda.get_device_capability()
print('GPU:', torch.cuda.get_device_name(0), '| compute capability:', cc)
build = subprocess.run(
    'cd /content/h3d/hy3dgen/texgen/custom_rasterizer && pip -q install .',
    shell=True, capture_output=True, text=True,
    env={**__import__('os').environ, 'TORCH_CUDA_ARCH_LIST': f'{cc[0]}.{cc[1]}'})
print('rasterizer build:', 'OK' if build.returncode == 0 else 'FAILED (will use fallback)')

# Fallback: Mew3D's pure-PyTorch rasterizer, in case the build fails on a future Colab image.
!wget -q -O /content/soft_raster.py https://raw.githubusercontent.com/Imtiaj-Sajin/mew3d/main/mew3d/core/soft_raster.py
print('SETUP DONE')

In [ ]:
# 2) Load the paint pipeline (T4's 16GB fits it without offloading)
import sys, torch
sys.path.insert(0, '/content')

try:
    import custom_rasterizer, torch as _t
    pos = _t.tensor([[[-1.,-1.,0.,1.],[1.,-1.,0.,1.],[-1.,1.,0.,1.]]], device='cuda')
    tri = _t.tensor([[0,1,2]], dtype=_t.int32, device='cuda')
    f, _ = custom_rasterizer.rasterize(pos, tri, (32, 32)); _t.cuda.synchronize()
    assert bool((f > 0).any())
    print('rasterizer: compiled extension')
except Exception as e:
    print('compiled rasterizer unusable (', e, ') -> pure-PyTorch fallback')
    import types, soft_raster
    shim = types.ModuleType('custom_rasterizer')
    shim.rasterize, shim.interpolate = soft_raster.rasterize, soft_raster.interpolate
    shim._MEW3D_SHIM = True
    sys.modules['custom_rasterizer'] = shim

from hy3dgen.texgen import Hunyuan3DPaintPipeline
paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
print('PAINT PIPELINE READY')

In [ ]:
# 3) Serve  ->  prints the public URL to paste into your PC
import nest_asyncio, threading, tempfile, uvicorn, trimesh, re, subprocess, time
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import FileResponse
from PIL import Image
nest_asyncio.apply()

app = FastAPI()
lock = threading.Lock()

@app.get('/health')
def health():
    return {'ok': True, 'service': 'mew3d-texture', 'gpu': torch.cuda.get_device_name(0),
            'busy': lock.locked()}

@app.post('/texture')
def texture(mesh: UploadFile = File(...), image: UploadFile = File(...)):
    with lock:
        mp = tempfile.NamedTemporaryFile(suffix='.glb', delete=False).name
        ip = tempfile.NamedTemporaryFile(suffix='.png', delete=False).name
        open(mp, 'wb').write(mesh.file.read())
        open(ip, 'wb').write(image.file.read())
        m = trimesh.load(mp, force='mesh')
        t0 = time.time()
        out = paint(m, image=Image.open(ip).convert('RGBA'))
        op = tempfile.NamedTemporaryFile(suffix='.glb', delete=False).name
        out.export(op)
        print(f'textured {len(m.faces)} faces in {time.time()-t0:.0f}s')
        return FileResponse(op, media_type='model/gltf-binary', filename='textured.glb')

threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000,
                                            log_level='warning'), daemon=True).start()
time.sleep(3)

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000',
                         '--no-autoupdate'], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
pattern = re.compile(r'https://(?!api\.)[a-z0-9]+(?:-[a-z0-9]+)+\.trycloudflare\.com')
for line in proc.stdout:
    hit = pattern.search(line)
    if hit:
        print('\n' + '=' * 62)
        print('  MEW3D_TEXTURE_URL=' + hit.group(0))
        print('  paste that into your PC .env, then restart the studio')
        print('=' * 62 + '\n')
        break
    if 'failed to request quick Tunnel' in line:
        print('tunnel failed - re-run this cell to retry')
        break